# Setup
including loading packages and defining helper plotting functions

NOTE: necessary to run if you are running anything else in this notebook

In [13]:
%load_ext autoreload
%autoreload 2

import argparse
import sys
import os

import pandas as pd
import seaborn as sns
from matplotlib.patches import Rectangle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from PIL import Image, ImageDraw, ImageFont

from src import secondlevel_group_glm
from src import secondlevel_group_encoding
from src import secondlevel_individual_encoding
from src import helpers
from src import plotting_helpers

load = True #set load=True to load analysis files (need to have downloaded them beforehand). False will run all analyses before plotting them (will take longer)
generate_subfigures = True # will take much longer than downloading the subfigures to use in making the main figures
base_dir = '../' #automatically uses this project directory, can be set to a different location

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
def get_default_args(base_dir):
	args_dict = { 
				'model': None,
				'task':	'sherlock',
				'mask':	'ISC',
				'perc_top_voxels':	5,
				'space': 'MNI152NLin2009cAsym',
				'smoothing_fwhm': 3.0,
				'chunklen': 120,
				'population': 'NT',
				'feature_of_interest': None,
				'ISC_type':'leave_one_out',
				'dir': base_dir,
				'out_dir': os.path.join(base_dir,'analysis'),
				'figure_dir': os.path.join(base_dir,'figures','subfigures')
			}
	return args_dict
def construct_args(args_dict):
	parser = argparse.ArgumentParser()
	parser.add_argument('--model','-model',type=str,default=args_dict['model'])
	parser.add_argument('--task','-task',type=str,default=args_dict['task'])
	parser.add_argument('--mask','-mask',type=str, default=args_dict['mask']) #the mask that contains all masks of interest (overarching mask )
	parser.add_argument('--perc-top-voxels','-perc-top-voxels',type=int,default=args_dict['perc_top_voxels'])
	parser.add_argument('--space','-space',type=str,default=args_dict['space'])
	parser.add_argument('--smoothing-fwhm','-smoothing-fwhm',type=float,default=args_dict['smoothing_fwhm'])
	parser.add_argument('--chunklen','-chunklen',type=int,default=args_dict['chunklen'])
	parser.add_argument('--population','-population',type=str,default=args_dict['population'])
	parser.add_argument('--feature-of-interest','-feature-of-interest',type=str,default=args_dict['feature_of_interest'])
	parser.add_argument('--ISC-type','-ISC-type',type=str,default=args_dict['ISC_type'])

	parser.add_argument('--dir', '-dir', type=str,
						default=args_dict['dir'])
	parser.add_argument('--out_dir', '-output', type=str,
						default=args_dict['out_dir'])
	parser.add_argument('--figure_dir', '-figures', type=str,
						default=args_dict['figure_dir'])
	def filter_jupyter_args(args):
		# Filter out any args starting with --f= (Jupyter passes --f=<file>)
		return [arg for arg in args if not arg.startswith('--f=')]
	filtered_args = filter_jupyter_args(sys.argv)
	args = parser.parse_args(filtered_args[1:])  # [1:] to skip the script name
	return args

def plot_secondlevel_group_glm(base_dir,task,cmap='gray_inferno'):
	args_dict = get_default_args(base_dir)
	args_dict['task']=task
	args_dict['mask']=None
	args = construct_args(args_dict)

	### GROUP ANALYSIS
	secondlevel_group = secondlevel_group_glm.SecondLevelGroup(args)
	secondlevel_group.do_stats = False

	secondlevel_group.compile_data()
	# secondlevel_group.plot_weights()
	secondlevel_group.plot_zscores(vmin=0,FDR_correction=True, threshold=0.01,symmetric_cbar=False,cmap=cmap) #
def plot_secondlevel_group_encoding(base_dir,model,plotting,feature_names,cmap='gray_inferno',diff_cmap='',mask='ISC',load=False,vmax=None):
	args_dict = get_default_args(base_dir)
	args_dict['model']=model
	args_dict['mask'] = mask
	args_dict['mask_name'] = mask
	if(model=='correlation'):
		args_dict['smoothing_fwhm'] = 6.0

	args = construct_args(args_dict)

	### GROUP ANALYSIS
	secondlevel_group = secondlevel_group_encoding.SecondLevelGroup(args)
	secondlevel_group.do_stats = False
	secondlevel_group.explainable_variance_mask = False
	secondlevel_group.feature_names = feature_names
	secondlevel_group.scale_by=''#'total_variance'
	if(not load):
		secondlevel_group.compile_data(measures=['performance']+plotting) #always need to do performance to get proper brain shape and affine
	if('preference_map' in plotting):
		secondlevel_group.compute_preference_maps(restricted=False)
	secondlevel_group.save_results()
	if('performance' in plotting):
		title = ''
		if(model=='correlation'):
			secondlevel_group.permutation_brainiak(load=load)
			title='Intersubject Correlation'
		else:
			secondlevel_group.stats_wilcoxon()
			title = 'Joint Encoding Model Performance'
		secondlevel_group.plot_performance('stats',threshold=0.00001,vmin=0,vmax=vmax,cmap=cmap,title=title)
	if('ind_product_measure' in plotting):
		secondlevel_group.stats_wilcoxon()
		secondlevel_group.plot_ind_product_measure('stats',threshold=0.00001,vmin=0,vmax=vmax,cmap=cmap)
	if('added_variance' in plotting):
		secondlevel_group.plot_added_variance('raw',threshold=0.00001,vmin=0,vmax=vmax,cmap=cmap)
	if('unique_variance' in plotting):
		secondlevel_group.plot_unique_variance('raw',threshold=0.00001,vmin=0,vmax=vmax,cmap=cmap)
	if('spatial_correlation' in plotting):
		secondlevel_group.plot_spatial_correlation('raw',threshold=None,vmin=None,vmax=vmax,cmap=cmap)
	if('ind_feature_performance' in plotting):
		secondlevel_group.plot_ind_feature_performance('raw',threshold=0.01,vmin=0,vmax=vmax,cmap=cmap)
	if('preference_map' in plotting):
		secondlevel_group.plot_preference_maps(label='both')
	if('difference' in plotting):
		secondlevel_group.plot_difference('raw',threshold=0.01,cmap=diff_cmap,vmax=vmax)
def get_secondlevel_individual(base_dir, model):
	args_dict = get_default_args(base_dir)
	args_dict['model']=model
	args = construct_args(args_dict)

	secondlevel_ind = secondlevel_individual_encoding.SecondLevelIndividual(args)
	secondlevel_ind.localizer_contrasts = {'SIpointlights':{'interact&no_interact','interact-no_interact'},'social interaction':{'interact&no_interact','interact-no_interact'},'language':{'intact-degraded'}}
	
	## specifying the names of each mask
	MT = ['MT']
	ISC = ['ISC']
	STS = ['pSTS','aSTS']#'frontal_language']
	language =['pTemp','aTemp']#,'frontal_language']
	language_ROI_names = ['pTemp','aTemp']#,'frontal']
	secondlevel_ind.localizer_masks = {'interact&no_interact':MT,'interact-no_interact':STS,'intact-degraded':language, 'degraded-intact':STS,
							'motion pointlights':MT,'SI pointlights':STS, 'language':language, 'DMN':STS,
							'social interaction':STS,'social':STS,
       						'GPT2':STS,'sbert':language,'sbert+word2vec':language,'word2vec':language,'hubert':language,
							'motion':MT,'num_agents':STS, 'alexnet':STS,'alexnet':STS,
							'valence':STS,'face':STS,'mentalization':STS, 'arousal':STS,
							'speaking':STS,'indoor_outdoor':STS,
							'turn_taking':STS,'written_text':STS,'music':STS,'none':STS}
	secondlevel_ind.MT = MT
	secondlevel_ind.ISC = ISC
	secondlevel_ind.STS = STS
	secondlevel_ind.language = language
	secondlevel_ind.language_ROI_names = language_ROI_names
	return secondlevel_ind

In [15]:
#load the default arguments for easy access
args_dict = get_default_args(base_dir)
subject = 'sub-33' #specifying representative subject #sub-26, sub-39 are other options

In [ ]:
#helper functions to compile the feature similarity results and plot them
def compile_featurespace_correlation_results(secondlevel_ind,latent_dim=None):
    features = secondlevel_ind.feature_names
    filepath = os.path.join(secondlevel_ind.dir,'analysis','FeatureSpaceCorrelation')
    results_matrix = np.full((len(features),len(features)), np.nan)
    for i,feature1 in enumerate(features):
        for j,feature2 in enumerate(features):
            if(feature1==feature2):
                continue
            if(latent_dim!=None):
                fname = '-'.join([feature1,feature2])+'_latent_dim-'+str(latent_dim)+'.csv'
            else:
                fname = '-'.join([feature1,feature2])+'.csv'
            try:
                result = float(pd.read_csv(os.path.join(filepath,fname),header=None).iloc[0,0])
            except FileNotFoundError:
                result = np.nan
            if(np.isnan(results_matrix[i][j])):
                results_matrix[i][j] = result
            if(np.isnan(results_matrix[j][i])):
                results_matrix[j][i] = result
    return results_matrix
def plot_vector_heatmap(ax,matrix,cmap,vmin=0,vmax=1):
    #draw each cell as a vector patch (pcolormesh) instead of an embedded raster image (imshow), so the saved pdf is fully vectorized
    n_rows,n_cols = matrix.shape
    mesh = ax.pcolormesh(np.arange(n_cols+1)-0.5,np.arange(n_rows+1)-0.5,np.ma.masked_invalid(matrix),
                         cmap=cmap,vmin=vmin,vmax=vmax,edgecolors='face',linewidth=0.5,rasterized=False) #face-colored edges hide hairline seams between cells
    #match imshow's orientation and aspect so ticks/text positions are unchanged
    ax.set_xlim(-0.5,n_cols-0.5)
    ax.set_ylim(n_rows-0.5,-0.5)
    ax.set_aspect('equal')
    return mesh
def plot_featurespace_correlation_layerwise(results_matrix,features,output_filename,cmap,width_mm=180,fontsize=6):
    #layerwise matrix drawn at its printed size: width_mm wide, all text at fontsize (pt), saved as an editable pdf
    from matplotlib import transforms
    from matplotlib.lines import Line2D
    plt.rcParams.update({'font.family': 'Arial','font.size': fontsize,'pdf.fonttype': 42})
    results_matrix = results_matrix.copy()
    results_matrix[np.triu_indices(results_matrix.shape[0],0)] = np.nan
    n = len(features)
    width_in = width_mm/25.4
    fig = plt.figure(figsize=(width_in,width_in*0.93))
    ax = fig.add_axes([0.08,0.11,0.80,0.80/0.93]) #square matrix filling the width, with room for the labels and colorbar
    cax = plot_vector_heatmap(ax,results_matrix,cmap,vmin=0,vmax=1)
    sns.despine(ax=ax,left=True,bottom=True)

    #DNN layers are labelled by their layer number, with the model name on a bracket; everything else by its name
    group_names = {'alexnet':'AlexNet','hubert':'HuBERT','gpt2':'GPT-2'}
    other_labels = {'word2vec_chunk120':'word2vec','num_agents':'num. agents','turn_taking':'turn taking','written_text':'written text'}
    groups = [name.split('_')[0].lower() if ('_layer' in name and name.split('_')[0].lower() in group_names) else None for name in features]
    tick_labels = [name.split('_layer')[1] if group else other_labels.get(name,name.replace('_',' ')) for name,group in zip(features,groups)]
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(tick_labels,rotation=90,fontsize=fontsize)
    ax.set_yticklabels(tick_labels,fontsize=fontsize)
    ax.tick_params(axis='both',which='both',length=0,pad=1.5)

    #brackets with the model names, a fixed distance (in points) outside the layer numbers
    offset_pt = 10
    y_side = transforms.blended_transform_factory(ax.transAxes,ax.transData) + transforms.ScaledTranslation(-offset_pt/72,0,fig.dpi_scale_trans)
    x_side = transforms.blended_transform_factory(ax.transData,ax.transAxes) + transforms.ScaledTranslation(0,-offset_pt/72,fig.dpi_scale_trans)
    for group,name in group_names.items():
        idx = [i for i,g in enumerate(groups) if g==group]
        if len(idx)==0:
            continue
        lo,hi = idx[0]-0.4,idx[-1]+0.4
        mid = (idx[0]+idx[-1])/2
        ax.add_line(Line2D([0,0],[lo,hi],transform=y_side,color='black',linewidth=0.5,clip_on=False))
        ax.text(0,mid,name+' ',transform=y_side,ha='right',va='center',fontsize=fontsize)
        ax.add_line(Line2D([lo,hi],[0,0],transform=x_side,color='black',linewidth=0.5,clip_on=False))
        ax.text(mid,0,' \n'+name,transform=x_side,ha='center',va='top',fontsize=fontsize)

    cbar = fig.colorbar(cax,ax=ax,shrink=0.4,aspect=15,location='right',anchor=(0,0.85),pad=0.01)
    cbar.set_label('Canonical correlation coefficient',fontsize=fontsize)
    cbar.ax.tick_params(labelsize=fontsize,width=0.5,length=2)
    cbar.outline.set_linewidth(0.5)
    cbar.solids.set_rasterized(False) #matplotlib rasterizes colorbars by default; keep it vector
    cbar.solids.set_edgecolor('face') #hide hairline seams between colorbar segments
    fig.savefig(output_filename,dpi=300,transparent=True,bbox_inches='tight',pad_inches=0.02) #trimming only removes empty margin, so the width stays <= width_mm
    plt.close(fig)

    #the plotted matrix (lower triangle) with its row and column labels, for the source data
    plotted_df = pd.DataFrame(results_matrix, index=features, columns=features)
    return fig, plotted_df
def plot_featurespace_correlation(secondlevel_ind,results_matrix,output_filename,average_over_layers=True,width_mm=180,fontsize=6):
    #width_mm, fontsize: printed width (mm) and size of all text (pt) of the layerwise matrix (average_over_layers=False)
    
    cmap =  plotting_helpers.get_cmaps()['matrix_green']
    features = secondlevel_ind.feature_names
    if average_over_layers:
        font_size = 0.6*len(features)
        font_size = 50
        print(font_size)
        # anchor = (0.7,-1.6)
        anchor = (0,0.5)
    else:
        font_size = 1.25*len(features)
        # anchor = (0.8,-0.75)
        anchor = (-0.75,0.8)
    plt.rcParams.update({'font.family': 'Arial','font.size': font_size,'pdf.fonttype': 42}) #fonttype 42 keeps text editable in the pdf

    if average_over_layers:
        results_df = pd.DataFrame(results_matrix, columns=features)
        results_df['feature'] = [feature+'_layer' if 'layer' not in feature else feature for feature in features]
        results_df['model'] = [label.split('_layer')[0] for label in results_df['feature']]
        results_df['layer'] = [label.split('layer')[1] for label in results_df['feature']]
        results_df['output'] = [label.split('_layer')[1] for label in results_df['feature']]
        results_df['model_output'] = [label.split('_layer')[0] if 'layer' in label.split('_')[1] else label.split('_')[0]+'_'+label.split('_')[1] for label in results_df['feature']]
        results_df.columns = results_df['model_output'].tolist() + ['feature','model','layer','output','model_output']
        
        #average over rows of same model_output, then average over columns of same name
        results_df=results_df.groupby(by='model_output', axis=0).mean(numeric_only=True)
        results_df=results_df.T.groupby(level=0).mean().T
        #populate a numpy matrix to display
        features_no_layer = [feature if 'layer' not in feature else feature.split('_layer')[0] for feature in features]
        # features_no_layer = [feature if 'eps0.3' not in feature else feature.split('_eps0.3')[0] for feature in features_no_layer]
        features = list(dict.fromkeys(features_no_layer)) #get rid of duplicates
        results_matrix = np.full((len(features),len(features)), np.nan)
        for i,feature1 in enumerate(features):
            for j,feature2 in enumerate(features):
                if(~np.isnan(results_df[feature1][feature2])):
                    results_matrix[i][j] = results_df[feature1][feature2]
                    results_matrix[j][i]  = results_df[feature1][feature2]

        results_matrix[np.triu_indices(results_matrix.shape[0],0)] = np.nan
        fig = plt.figure(figsize=(len(features)*1.5,len(features)*1.5))

        fig = plt.figure(figsize=(len(features)*2.5,len(features)*2.5))

        ax = fig.add_subplot(111)
        cax = plot_vector_heatmap(ax,results_matrix,cmap,vmin=0,vmax=1)
        sns.despine(left=True,bottom=True)

        plt.title('')
        #Set the labels of each row and column, skipping the first label of the rows and the last label of the columns (no data there)
        ax.set_xticks(range(0,len(features)))
        ax.set_yticks(range(0,len(features)))
        feature_labels = [feature.replace('_',' ').replace(' layer','').replace('1sent','') for feature in features]
        feature_labels = [feature.replace(' ','_') for feature in feature_labels]
        feature_labels_dict = {'alexnet_eps0.3':'image','motion':'motion','hubert_chunk120':'speech','gpt2_chunk120':"multi-sentence",'word2vec_chunk120':"word",'social':'social\ninteraction','SimCLR_attention':'vision_attention','SimCLR_embedding':'vision_emb'}
        feature_labels_final = [feature_labels_dict.get(feature.lower(),feature) for feature in feature_labels]
        ax.set_xticklabels(feature_labels_final[:-1]+[''], minor=False, rotation=90)
        ax.set_yticklabels(['']+feature_labels_final[1:], minor=False)

        print(feature_labels)

        # Apply colors to labels 
        colors_dict = plotting_helpers.get_colors_dict()
        for label, xticklabel, yticklabel in zip(feature_labels, ax.get_xticklabels(), ax.get_yticklabels()):
            print(label)
            color = colors_dict.get(label, 'black')  # Default to black if not in dict
            xticklabel.set_color(color)
            yticklabel.set_color(color)
        
        for i in range(len(features)):
            for j in range((len(features))):
                c = results_matrix[j,i]
                if(~np.isnan(c)):
                    ax.text(i, j, str(np.round(c,2)), va='center', ha='center',fontweight='light')
    else:
        return plot_featurespace_correlation_layerwise(results_matrix,features,output_filename,cmap,width_mm=width_mm,fontsize=fontsize)

    # Remove tick marks but keep labels
    ax.tick_params(axis='both', which='both', length=0)  # Set tick length to 0 
    
    cbar = fig.colorbar(cax,label='Canonical correlation coefficient',shrink=0.5,location='right',anchor = anchor)
    cbar.ax.yaxis.set_label_position('left')
    cbar.solids.set_rasterized(False) #matplotlib rasterizes colorbars by default; keep it vector
    cbar.solids.set_edgecolor('face') #hide hairline seams between colorbar segments
    # cbar.ax.yaxis.set_ticks_position('default')
    # Move label below the bar
    # cbar.ax.xaxis.set_label_position('bottom')
    # cbar.ax.xaxis.tick_bottom()  # Ensure ticks are also at the bottom

    plt.xticks(rotation=90)
    plt.savefig(output_filename,bbox_inches='tight',dpi=300)
    plt.close()

    #the plotted matrix (lower triangle) with its row and column labels, for the source data
    plotted_labels = [label.replace('\n',' ') for label in feature_labels_final] if average_over_layers else features
    plotted_df = pd.DataFrame(results_matrix, index=plotted_labels, columns=plotted_labels)
    return fig, plotted_df

In [77]:
def label_brain_plots(output_filename, final_image, labels_dict={}, desc_dict={}, feature_names=[], features_on_width=True,pad_prop=0,include_desc=True,include_ind_group_labels=True,color_dict=None):
    # add padding for the labels
    pad = 200
    if features_on_width:
        padding = (pad, pad, pad, pad)
    else:
        padding = (pad, pad, pad, 0)
    
    new_width = final_image.width + padding[0] + padding[2]
    new_height = final_image.height + padding[1] + padding[3]

    new_image = Image.new('RGBA', (new_width, new_height), (255, 255, 255, 0))
    new_image.paste(final_image, (padding[0], padding[1]))

    draw = ImageDraw.Draw(new_image)
    font = ImageFont.truetype("Arial.ttf", 85)
    label_font = ImageFont.truetype("Arial.ttf", 125)  # Larger font for 'Individual' and 'Group'
    width, height = final_image.size
    
    if features_on_width:
        portion = width // len(feature_names)
        centers = [(portion * i + portion // 2.35, 15) for i in range(len(feature_names))]
    else:
        portion = height // len(feature_names)
        centers = [(new_width // 2, (portion * i)-pad_prop*portion/3) for i in range(len(feature_names))]
    
    text_padding = 25
    if(len(feature_names)>1):
        for center, feature in zip(centers, feature_names):
            bbox = draw.textbbox((0, 0), labels_dict[feature], font=font)
            text_width = bbox[2] - bbox[0]
            text_height = bbox[3] - bbox[1]
            
            if features_on_width:
                text_x = center[0] - (text_width // 2) + pad
                text_y = center[1] + (text_height // 2)
                rectangle_x = center[0] - portion // len(feature_names)
                rectangle_coords = (
                    rectangle_x + pad,                # Left
                    text_y - text_padding,                     # Top
                    rectangle_x + (portion//len(feature_names)*2) +pad ,        # Right
                    text_y + text_padding*5      # Bottom
                    )
            else:
                text_x = center[0] - (text_width // 2)
                text_y = center[1] - (text_height // 2) + pad
                rectangle_x = center[0] - (text_width //2)
                rectangle_coords = (
                    rectangle_x - pad,                # Left
                    text_y - text_padding,                     # Top
                    rectangle_x + text_width+pad ,        # Right
                    text_y + text_padding*5      # Bottom
                    )
            
            draw.text((text_x,text_y), labels_dict[feature], fill="black", font=font)
            
            if(include_desc):
                bbox = draw.textbbox((0, 0), desc_dict[feature], font=font)
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
                
                if features_on_width:
                    text_x = center[0] - (text_width // 2) + pad
                    text_y = new_image.size[1] - text_height * 1.6
                else:
                    text_x = center[0] - text_width // 2
                    text_y = center[1]  + portion
                
                draw.text((text_x, text_y), desc_dict[feature], fill="black", font=font)
            
    
    if(include_ind_group_labels):
        if features_on_width:
            #rotated labels for labeling Individual and Group brain maps
            for y,text in zip((1,3),['Individual','Group']):
                bbox = draw.textbbox((0, 0), text, font=font)
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
                text_image = Image.new('RGBA', (text_width,text_height+20), (255, 255, 255, 0)) #transparent background
                draw = ImageDraw.Draw(text_image)
                text_position = (0,0)
                draw.text(text_position, text, font=font, fill="black")
                text_image = text_image.rotate(90, resample=Image.BICUBIC, expand=1)
                new_image.paste(text_image, (-15, (final_image.size[1]//4)*y), text_image)
        else:
        # Labels for 'Individual' and 'Group' along the top
            label_positions = [(0.24, 'Individual'), (0.7, 'Group')]
            for pos, text in label_positions:
                bbox = draw.textbbox((0, 0), text, font=label_font)
                text_width = bbox[2] - bbox[0]
                text_x = int(new_image.size[0] * pos - text_width / 2)
                text_y = new_height - (pad)
                draw.text((text_x, text_y), text, font=label_font, fill="black")

    new_image.save(output_filename)
    return new_image

# Main Figures
NOTE: you can run these if you cloned the GitHub repository and ran Setup (above). 

### Main Figure 1C (layer averaged feature similarity of specific vision and language features during movie)

In [ ]:
##MAIN
secondlevel_ind = get_secondlevel_individual(base_dir,'joint_chunk120')
#only display some of the features
secondlevel_ind.feature_names =  ['alexnet_eps0.3_layer'+str(layer) for layer in [1,2,3,4,5,6,7]] +['motion','social'] + ['hubert_chunk120_layer'+str(layer) for layer in [1,2,3,4,5,6,7,8,9,10,11,12]] + ['word2vec_chunk120'] + ['GPT2_chunk120_layer'+str(layer) for layer in [1,2,3,4,5,6,7,8,9,10,11,12]] 
results_matrix = compile_featurespace_correlation_results(secondlevel_ind,latent_dim=1)
output_filename = os.path.join(secondlevel_ind.dir,'figures','main',f'featurespace_correlation_layers_averaged.pdf')
_, plotted_matrix = plot_featurespace_correlation(secondlevel_ind,results_matrix,output_filename,average_over_layers=True)
print('layer averaged similarity matrix saved: ', output_filename)

## canonical correlation between each pair of feature spaces (layers averaged), as plotted
plotted_matrix.to_csv(os.path.join(secondlevel_ind.dir,'figures','main','sourcedata','Figure1C.csv'))


### Main Figure 2. Cross-subject encoding performance and joint feature encoding model performance group maps

In [97]:
images = []

argument_list = ['mask','chunklen'] #arguments to include in the filename
base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelGroup','performance')
post_filepath = '_'.join([argument.replace('_','') + '-'+str(args_dict[argument]) for argument in argument_list ])
images.append(os.path.join(base_filepath,f"sub-{args_dict['population']}_brain2brain_encoding_smoothingfwhm-3.0_{post_filepath}_measure-perf_stats.pdf"))

argument_list = ['smoothing_fwhm','mask','chunklen'] #arguments to include in the filename
base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelGroup','performance')
post_filepath = '_'.join([argument.replace('_','') + '-'+str(args_dict[argument]) for argument in argument_list ])
images.append(os.path.join(base_filepath,f"sub-{args_dict['population']}_encoding_model-joint_chunk120_{post_filepath}_measure-perf_stats.pdf"))

argument_list = ['smoothing_fwhm','mask','chunklen'] #arguments to include in the filename
base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelGroup','ind_product_measure')
post_filepath = 'sub-NT_encoding_model-joint_chunk120_'+ '_'.join([argument.replace('_','') + '-'+str(args_dict[argument]) for argument in argument_list ])
images.append(os.path.join(base_filepath,post_filepath+'_measure-ind_product_measure_stats_feature-alexnet_eps0.3_vmax-0.1.pdf'))

argument_list = ['smoothing_fwhm','mask','chunklen'] #arguments to include in the filename
base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelGroup','ind_product_measure')
post_filepath = 'sub-NT_encoding_model-joint_chunk120_'+ '_'.join([argument.replace('_','') + '-'+str(args_dict[argument]) for argument in argument_list ])
images.append(os.path.join(base_filepath,post_filepath+'_measure-ind_product_measure_stats_feature-GPT2_chunk120_vmax-0.1.pdf'))

output_filename = os.path.join(args_dict['dir'],'figures','main',f'encoding_model_group_maps.pdf')
final_image = plotting_helpers.compose_final_figure_pdf(output_filename,images,'',title=None,num_columns=2,row_padding_proportion=0.15,label_subplots=True)

print('intersubject correlation and encoding model brain maps saved:', output_filename)

intersubject correlation and encoding model brain maps saved: ../figures/main/encoding_model_group_maps.pdf


### Main Figure 3. Joint model variance decomposition

In [ ]:
## create subfigure of the representative preference maps
argument_list = ['smoothing_fwhm','chunklen']
images = []
images.append(os.path.join(args_dict['figure_dir'],'SecondLevelGroup','preference_map','sub-NT_encoding_model-joint_chunk120_smoothingfwhm-3.0_mask-ISC_chunklen-120_measure-preference1_map.pdf'))
for subject in ['sub-61']: #sub-33','sub-62''sub-51'
    base_filepath = subject+'_encoding_model-joint_chunk120_'+'_'.join([argument.replace('_','')+'-'+str(args_dict[argument]) for argument in argument_list])
    images.append(os.path.join(args_dict['figure_dir'],'SecondLevelIndividual','preference_map',base_filepath+'_measure-ind_product_measure_preference1_map_vent_lat.pdf'))
argument_list = ['smoothing_fwhm','chunklen','mask','model']
color_dict = {}

output_filename = os.path.join(args_dict['dir'],'figures','subfigures','SecondLevelIndividual',f'joint_chunk120_preference_map.pdf')
plotting_helpers.compose_final_figure_pdf(output_filename,images,'',title=None,num_columns=1)

## compile into full figure with the ROI barplot
images = []
argument_list = ['smoothing_fwhm','chunklen','mask']
base_filepath = 'sub-'+args_dict['population']+'_'+'_'.join([argument.replace('_','')+'-'+str(args_dict[argument]) for argument in argument_list])
images.append(os.path.join(args_dict['dir'],'figures','subfigures','SecondLevelIndividual',f'joint_chunk120_preference_map.pdf'))
images.append(os.path.join(args_dict['figure_dir'],'SecondLevelIndividual',base_filepath+'_model-joint_chunk120_glm_localizer_enc_response-ind_product_measure_perc_top_voxels-'+str(args_dict['perc_top_voxels'])+'__all_stacked.pdf'))
output_filename = os.path.join(args_dict['dir'],'figures','main',f'joint_product_measure.pdf')
plotting_helpers.compose_final_figure_pdf(output_filename,images,'',title=None,num_columns=2,label_subplots=True,panel_top_padding_mm=[0,4])
print('variance decomposition figure saved:',output_filename)

## each participant's product measure of each feature in each region (panel B; panel A is a brain map)
plotting_helpers.collect_source_data([os.path.join(args_dict['figure_dir'],'SecondLevelIndividual',base_filepath+'_model-joint_chunk120_glm_localizer_enc_response-ind_product_measure_perc_top_voxels-'+str(args_dict['perc_top_voxels'])+'__all_source_data.csv')],
                                     os.path.join(args_dict['dir'],'figures','main','sourcedata'),'Figure3B')

### Main Figure 4. Encoding weight similarity across regions

In [ ]:
## generated in SecondLevel Analyses.ipynb (Analyses of encoding model weights); the colorbar on the last matrix is shared
argument_list = ['smoothing_fwhm','chunklen','mask']
base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelIndividual','response_similarity','sub-'+args_dict['population']+'_'+'_'.join([argument.replace('_','')+'-'+str(args_dict[argument]) for argument in argument_list]))
panels = [base_filepath+f'_model-vislang_gpt2_chunk120_eps0.3_weights_response_similarity_perc_top_voxels-5-{axis}_similarity_matrix_{filepath_tag}'
          for axis,filepath_tag in [('motion','motion'),('alexnet','alexnet_eps0.3'),('hubert','hubert_chunk120'),('GPT2','GPT2_chunk120')]]
output_filename = os.path.join(args_dict['dir'],'figures','main','response_similarity_across_regions.pdf')
plotting_helpers.compose_final_figure_pdf(output_filename,[panel+'.pdf' for panel in panels],'',title=None,num_columns=4,label_subplots=False)
print('response similarity figure saved:',output_filename)

## each participant's response similarity for every region pair and feature space (one csv for the figure)
sourcedata_csv = os.path.join(args_dict['dir'],'figures','main','sourcedata','Figure4.csv')
pd.concat([pd.read_csv(panel+'_source_data.csv') for panel in panels]).to_csv(sourcedata_csv,index=False)
print('source data saved:',sourcedata_csv)

### Main Figure 5. Interpreting visual and language predictivity in ROIs

In [ ]:
from src import weight_pca_analysis
## max r² across the first 5 pooled PCs of each annotated feature in each region x model (computed in SecondLevel Analyses.ipynb)
## bars are colored when the feature is significantly correlated with one of those PCs (FDR q<0.05), gray otherwise; dashed lines are the 95th percentile of the permutation null
## features that are never significant are dropped from the x-axis
pca_args = argparse.Namespace(dir=args_dict['dir'],out_dir=args_dict['out_dir'],model='vislang_gpt2_chunk120_eps0.3',
                              smoothing_fwhm=args_dict['smoothing_fwhm'],chunklen=args_dict['chunklen'],mask=args_dict['mask'],
                              space=args_dict['space'],population=args_dict['population'])
weight_pca = weight_pca_analysis.WeightPCAAnalysis(pca_args)

plotting_helpers.figure_width_mm = 180 #final width of the saved pdf
plotting_helpers.figure_fontsize = 7 #pt, all text
weight_pca.plot_pooled_pca_summary(image_model='alexnet',compress_xaxis=True,mask_nonsig=False,figsize=(7.1,5))
## saves figures/main/pooled_pca_summary_nonsig_shown_alexnet.pdf

plotting_helpers.figure_width_mm = None
plotting_helpers.figure_fontsize = None

## the plotted values (max r² of each feature across the first 5 pooled PCs, with its significance) are saved next to the figure
plotting_helpers.collect_source_data([os.path.join(args_dict['dir'],'figures','main','pooled_pca_summary_nonsig_shown_alexnet.csv')],
                                     os.path.join(args_dict['dir'],'figures','main','sourcedata'),'Figure5')

# Supplementary Figures
NOTE: you can run these if you cloned the GitHub repository and ran Setup (above). 

### Supplementary Figure 1. Individual and group maps of localizer tasks

In [ ]:
## example participant (sub-33) and group localizer z-score maps (made in SecondLevel Analyses.ipynb).
## Each map is a 90 mm wide pdf with 6 pt editable text, so two columns make the 180 mm figure without rescaling
argument_list = ['smoothing_fwhm'] #arguments to include in the filename
contrasts = ['interact&no_interact','interact-no_interact','intact-degraded']

subject = 'sub-33'
base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelIndividual','glm_zscores')
post_filepath = subject+'_'+ '_'.join([argument.replace('_','') + '-'+str(args_dict[argument]) for argument in argument_list ])
individual_images = [os.path.join(base_filepath,post_filepath+'_measure-zscore_contrast-'+contrast+'.pdf') for contrast in contrasts]

base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelGroup','glm_zscores')
post_filepath = 'sub-'+args_dict['population']+'_'+ '_'.join([argument.replace('_','') + '-'+str(args_dict[argument]) for argument in argument_list ])
group_images = [os.path.join(base_filepath,post_filepath+'_measure-zscore_contrast-'+contrast+'.pdf') for contrast in contrasts]

all_images =[val for pair in zip(individual_images, group_images) for val in pair]
labels_dict = {'interact&no_interact':'Motion Responsive (interacting + not interacting)',
               'interact-no_interact':'Social Interaction Selective (interacting - not interacting)',
               'intact-degraded':'Language Selective (intact - degraded speech)'}

output_filename = os.path.join(args_dict['dir'],'figures','supplemental','glm_zscore_maps.pdf')
plotting_helpers.compose_final_figure_pdf(output_filename,all_images,'',title=None,num_columns=2,row_padding_proportion=0.05,
                                          row_titles=[labels_dict[contrast] for contrast in contrasts],column_titles=['Individual','Group'])
plotting_helpers.compress_pdf_images(output_filename) #brain renders stored as ~450 dpi JPEG to keep the file small
print('glm zscore maps saved:',output_filename)

### Supplementary Figure 2. Neural network layerwise feature similarity

In [ ]:
##SUPPLEMENTARY FIGURE
secondlevel_ind = get_secondlevel_individual(base_dir,'joint_chunk120')
results_matrix = compile_featurespace_correlation_results(secondlevel_ind,latent_dim=1)
secondlevel_ind.feature_names = secondlevel_ind.feature_names #+ ['scene_description_layer'+str(layer) for layer in [1,2,3,4,5,6,7,8,9,10,11,12]]
output_filename = os.path.join(secondlevel_ind.dir,'figures','supplemental',f'featurespace_correlation.pdf')
_, plotted_matrix = plot_featurespace_correlation(secondlevel_ind,results_matrix,output_filename,average_over_layers=False)
print('layerwise similarity matrix saved: ',output_filename)

## canonical correlation between each pair of layers/feature spaces, as plotted
plotted_matrix.to_csv(os.path.join(secondlevel_ind.dir,'figures','supplemental','sourcedata','SupplementaryFigure2.csv'))

### Supplementary Figure 3. Individual and group maps of individual feature predictivity (AlexNet, motion, speech, and sentence)

In [ ]:
## example participant (sub-33) and group maps of each feature space's product measure (made in SecondLevel Analyses.ipynb).
## Each map is a 90 mm wide pdf with 6 pt editable text, so two columns make the 180 mm figure without rescaling
subject = 'sub-33'
args_dict['model'] = 'joint_chunk120'
feature_names = ['alexnet_eps0.3','motion','hubert_chunk120','GPT2_chunk120']
measure = 'ind_product_measure'
argument_list = ['model','smoothing_fwhm','mask','chunklen'] #arguments to include in the filename

base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelIndividual','map')
post_filepath = subject+'_encoding_'+ '_'.join([argument.replace('_','') + '-'+str(args_dict[argument]) for argument in argument_list ])
individual_images = [os.path.join(base_filepath,post_filepath+'_measure-'+measure+'_feature-'+feature+'.pdf') for feature in feature_names]

base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelGroup',measure)
post_filepath = 'sub-'+args_dict['population']+'_encoding_'+ '_'.join([argument.replace('_','') + '-'+str(args_dict[argument]) for argument in argument_list ])
group_images = [os.path.join(base_filepath,post_filepath+'_measure-'+measure+'_stats_feature-'+feature+'_vmax-None.pdf') for feature in feature_names]

all_images =[val for pair in zip(individual_images, group_images) for val in pair]
labels_dict = {'alexnet_eps0.3':'Image Model',
               'motion':'Motion Model',
               'hubert_chunk120':'Speech Model',
               'GPT2_chunk120':'Multi-sentence Model'}

output_filename = os.path.join(args_dict['dir'],'figures','supplemental','joint_ind_feature_product_measure_maps.pdf')
plotting_helpers.compose_final_figure_pdf(output_filename,all_images,'',title=None,num_columns=2,row_padding_proportion=0.05,
                                          row_titles=[labels_dict[feature] for feature in feature_names],column_titles=['Individual','Group'])
plotting_helpers.compress_pdf_images(output_filename) #brain renders stored as ~450 dpi JPEG to keep the file small
print('individual and group maps of individual feature predictivity saved:', output_filename)

### Supplementary Figure 4. All individual preference maps

In [121]:
## every participant's preference map (made in SecondLevel Analyses.ipynb), 12 participants per page, drawn at the printed size:
## width_mm wide, all text at fontsize (pt), brain images at dpi. Each map's own "Participant N" title is cropped off and redrawn as text
from matplotlib.patches import Patch
width_mm, fontsize, dpi = 180, 6, 450
num_columns, per_page = 3, 12

secondlevel_ind = get_secondlevel_individual(base_dir,'joint_chunk120')
argument_list = ['smoothing_fwhm','chunklen'] #arguments to include in the filename
color_dict = {label:secondlevel_ind.colors_dict[feature] for label,feature in
              [('image','alexnet_eps0.3'),('motion','motion'),('speech','hubert_chunk120'),('word','word2vec_chunk120'),('sentence','GPT2_chunk120')]}

def load_without_title(file):
    #drop the title rows at the top of the map (everything above the first fully transparent gap below it)
    image = Image.open(file).convert('RGBA')
    alpha = np.asarray(image)[:,:,3]
    content_rows = alpha.max(axis=1) > 0
    first = np.argmax(content_rows)
    gap = first + np.argmax(~content_rows[first:])
    image = image.crop((0, gap + np.argmax(content_rows[gap:]), image.width, image.height))
    background = Image.new('RGBA', image.size, 'white')
    background.alpha_composite(image)
    return background.convert('RGB')

subjects = secondlevel_ind.subjects['sherlock']
for page_start in range(0, len(subjects), per_page):
    page_subjects = subjects[page_start:page_start+per_page]
    images = []
    for subject in page_subjects:
        base_filepath = subject+'_encoding_model-joint_chunk120_'+'_'.join([argument.replace('_','')+'-'+str(args_dict[argument]) for argument in argument_list])
        images.append(load_without_title(os.path.join(args_dict['figure_dir'],'SecondLevelIndividual','preference_map',base_filepath+'_measure-ind_product_measure_preference1_map_vent_med_lat.png')))

    #layout in inches: equal columns, a title line above each map, a legend line at the bottom
    width_in = width_mm/25.4
    col_w = width_in/num_columns
    img_h = col_w*images[0].height/images[0].width*0.92 #maps drawn at 92% of the column width
    title_h, row_gap, legend_h = 2.2*fontsize/72, 0.06, 3*fontsize/72
    num_rows = int(np.ceil(per_page/num_columns)) #every page has the same height
    height_in = num_rows*(title_h+img_h) + (num_rows-1)*row_gap + legend_h
    plt.rcParams.update({'font.family':'Arial','font.size':fontsize,'pdf.fonttype':42})
    fig = plt.figure(figsize=(width_in,height_in))
    for i,(subject,image) in enumerate(zip(page_subjects,images)):
        row, col = divmod(i, num_columns)
        top = height_in - row*(title_h+img_h+row_gap)
        #downsample to the printed resolution
        target_w = int(round(col_w*0.92*dpi))
        image = image.resize((target_w, int(round(target_w*image.height/image.width))), Image.LANCZOS)
        ax = fig.add_axes([(col*col_w + col_w*0.04)/width_in, (top-title_h-img_h)/height_in, col_w*0.92/width_in, img_h/height_in])
        ax.imshow(np.asarray(image), interpolation='none')
        ax.axis('off')
        fig.text((col+0.5)*col_w/width_in, (top-title_h*0.45)/height_in, 'Participant '+str(page_start+i+1), ha='center', va='center', fontsize=fontsize)
    fig.legend(handles=[Patch(facecolor=color, edgecolor='black', linewidth=0.4, label=label) for label,color in color_dict.items()],
               loc='lower center', bbox_to_anchor=(0.5,0), ncol=len(color_dict), frameon=False, fontsize=fontsize,
               handlelength=1.6, handleheight=0.9, columnspacing=1.5, handletextpad=0.5)
    output_filename = os.path.join(args_dict['dir'],'figures','supplemental',f'joint_pref_maps_{page_start//per_page+1}.jpg')
    fig.savefig(output_filename, dpi=dpi, facecolor='white', pil_kwargs={'quality':90}) #jpeg keeps each page ~3 MB (png ~10 MB) with no visible loss at this dpi
    plt.close(fig)
    print('saved:', output_filename, '(%.0f x %.0f mm)'%(width_mm, height_in*25.4))

saved: ../figures/supplemental/joint_pref_maps_11.png
saved: ../figures/supplemental/joint_pref_maps_23.png
saved: ../figures/supplemental/joint_pref_maps_33.png


### Supplementary Figure 5. Responses to the conditions of the localizer experiments in motion, social interaction, and language regions

In [ ]:
## bars and individual participant points generated in SecondLevel Analyses.ipynb (Supplementary ROI analyses)
argument_list = ['smoothing_fwhm','chunklen','mask']
base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelIndividual','sub-'+args_dict['population']+'_'+'_'.join([argument.replace('_','')+'-'+str(args_dict[argument]) for argument in argument_list]))
images = [f'{base_filepath}_model-_glm_localizer_glm_response_perc_top_voxels-{args_dict["perc_top_voxels"]}_glm_response_bar_points.pdf']
output_filename = os.path.join(args_dict['dir'],'figures','supplemental','localizer_glm_responses.pdf')
plotting_helpers.compose_final_figure_pdf(output_filename,images,'',title=None,num_columns=1)
print('localizer responses figure saved:',output_filename)

## per-participant values plotted in each panel
sourcedata_dir = os.path.join(args_dict['dir'],'figures','supplemental','sourcedata')
plotting_helpers.collect_source_data([f'{base_filepath}_model-_glm_localizer_glm_response_perc_top_voxels-{args_dict["perc_top_voxels"]}_glm_response_source_data.csv'],sourcedata_dir,'SupplementaryFigure5')

### Supplementary Figure 6. Encoding model performance in ROIs (A. cross-subject encoding, B. vision vs. language predictivity in the joint model, C. with SimCLR as the image model)

In [ ]:
## bars and individual participant points generated in SecondLevel Analyses.ipynb (Supplementary ROI analyses)
argument_list = ['smoothing_fwhm','chunklen','mask']
base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelIndividual','sub-'+args_dict['population']+'_'+'_'.join([argument.replace('_','')+'-'+str(args_dict[argument]) for argument in argument_list]))
perc = str(args_dict['perc_top_voxels'])
images = [base_filepath+'_model-joint_chunk120_glm_localizer_enc_response-performance_perc_top_voxels-'+perc+'_bar_points__for_cross_subject_.pdf',
          base_filepath+'_model-joint_chunk120_glm_localizer_enc_response-ind_product_measure_perc_top_voxels-'+perc+'_bar_points_vision_vs_language_.pdf',
          base_filepath+'_model-joint_chunk120_simclr_glm_localizer_enc_response-ind_product_measure_perc_top_voxels-'+perc+'_bar_points_vision_vs_language_.pdf']
output_filename = os.path.join(args_dict['dir'],'figures','supplemental','cross_subject_vis_lang.pdf')
plotting_helpers.compose_final_figure_pdf(output_filename,images,'',title=None,num_columns=1,label_subplots=True)
print('additional ROI analyses figure saved:',output_filename)

## per-participant values plotted in each panel
sourcedata_dir = os.path.join(args_dict['dir'],'figures','supplemental','sourcedata')
plotting_helpers.collect_source_data([base_filepath+'_model-joint_chunk120_glm_localizer_enc_response-performance_perc_top_voxels-'+perc+'__for_cross_subject__source_data.csv', #outliers already excluded
                                       base_filepath+'_model-joint_chunk120_glm_localizer_enc_response-ind_product_measure_perc_top_voxels-'+perc+'_vision_vs_language__source_data.csv',
                                       base_filepath+'_model-joint_chunk120_simclr_glm_localizer_enc_response-ind_product_measure_perc_top_voxels-'+perc+'_vision_vs_language__source_data.csv'],
                                      sourcedata_dir,'SupplementaryFigure6')

### Supplementary Figure 7. Voxel-wise image (AlexNet) and sentence (GPT-2) product measures in ROIs

In [ ]:
## generated in SecondLevel Analyses.ipynb (Supplementary ROI analyses)
images = [os.path.join(args_dict['figure_dir'],'SecondLevelIndividual','spatial_correlation','alexnet_vs_gpt2_scatter_voxelwise_comparisons.pdf')]
output_filename = os.path.join(args_dict['dir'],'figures','supplemental','alexnet_vs_gpt2_voxelwise_comparison.pdf')
plotting_helpers.compose_final_figure_pdf(output_filename,images,'',title=None,num_columns=1)
print('voxelwise AlexNet and GPT-2 scatterplot saved:',output_filename)

## per-participant values plotted in each panel
sourcedata_dir = os.path.join(args_dict['dir'],'figures','supplemental','sourcedata')
plotting_helpers.collect_source_data([images[0].replace('.pdf','_source_data.csv')],sourcedata_dir,'SupplementaryFigure7')

### Supplementary Figure 8. ROI stacked bar plots of encoding model performance using A) AlexNet and B) SimCLR as the image model

In [ ]:
## generated in SecondLevel Analyses.ipynb (Supplementary ROI analyses)
argument_list = ['smoothing_fwhm','chunklen','mask']
base_filepath = os.path.join(args_dict['figure_dir'],'SecondLevelIndividual','sub-'+args_dict['population']+'_'+'_'.join([argument.replace('_','')+'-'+str(args_dict[argument]) for argument in argument_list]))
images = [base_filepath+'_model-'+model+'_glm_localizer_enc_response-ind_product_measure_perc_top_voxels-'+str(args_dict['perc_top_voxels'])+'__sep_post_ant_stacked.pdf'
          for model in ['joint_chunk120','joint_chunk120_simclr']]
output_filename = os.path.join(args_dict['dir'],'figures','supplemental','alexnet_and_simclr_joint_model_product_measure.pdf')
plotting_helpers.compose_final_figure_pdf(output_filename,images,'',title=None,num_columns=1,label_subplots=True)
print('AlexNet and SimCLR joint model figure saved:',output_filename)

## per-participant values plotted in each panel
sourcedata_dir = os.path.join(args_dict['dir'],'figures','supplemental','sourcedata')
plotting_helpers.collect_source_data([base_filepath+'_model-'+model+'_glm_localizer_enc_response-ind_product_measure_perc_top_voxels-'+str(args_dict['perc_top_voxels'])+'__sep_post_ant_source_data.csv'
                                       for model in ['joint_chunk120','joint_chunk120_simclr']],sourcedata_dir,'SupplementaryFigure8')

### Supplementary Figure 9. Simplified joint encoding model performance in the full movie vs. only timepoints with speaking

In [ ]:
## generated in SecondLevel Analyses.ipynb (Speech-only evaluation)
images = [os.path.join(args_dict['figure_dir'],'subset_eval','roi_summary','sub-NT_encoding_model-vislang_gpt2_chunk120_eps0.3_smoothingfwhm-3.0_mask-ISC_chunklen-120_ind_product_measure_full_vs_high_speaking.pdf')]
output_filename = os.path.join(args_dict['dir'],'figures','supplemental','speech_masked_eval.pdf')
plotting_helpers.compose_final_figure_pdf(output_filename,images,'',title=None,num_columns=1)
print('speech-only evaluation figure saved:',output_filename)

## per-participant values plotted in each panel
sourcedata_dir = os.path.join(args_dict['dir'],'figures','supplemental','sourcedata')
plotting_helpers.collect_source_data([images[0].replace('.pdf','_source_data.csv')],sourcedata_dir,'SupplementaryFigure9')

### Supplementary Figure 10. Cumulative variance explained by the PCs of each feature space

In [ ]:
## generated in SecondLevel Analyses.ipynb (Dimensionality of the feature spaces)
images = [os.path.join(args_dict['figure_dir'],'IntrinsicDimensionality','intrinsic_dim_vislang_gpt2_chunk120.pdf')]
output_filename = os.path.join(args_dict['dir'],'figures','supplemental','intrinsic_dim_vislang_gpt2_chunk120.pdf')
plotting_helpers.compose_final_figure_pdf(output_filename,images,'',title=None,num_columns=1)
print('feature space dimensionality figure saved:',output_filename)

## per-participant values plotted in each panel
sourcedata_dir = os.path.join(args_dict['dir'],'figures','supplemental','sourcedata')
plotting_helpers.collect_source_data([images[0].replace('.pdf','_source_data.csv')],sourcedata_dir,'SupplementaryFigure10')

### Supplementary Figure 11. Correlations between the time series of the PCA of pooled voxelwise encoding model weights and the movie features

In [ ]:
from src import weight_pca_analysis
## squared correlation of each annotated feature with each of the first 5 pooled PCs in each region x model (computed in SecondLevel Analyses.ipynb)
## significant correlations (FDR q<0.05) are drawn solid with asterisks, the rest faded; dashed lines are the 95th percentile of the permutation null
pca_args = argparse.Namespace(dir=args_dict['dir'],out_dir=args_dict['out_dir'],model='vislang_gpt2_chunk120_eps0.3',
                              smoothing_fwhm=args_dict['smoothing_fwhm'],chunklen=args_dict['chunklen'],mask=args_dict['mask'],
                              space=args_dict['space'],population=args_dict['population'])
weight_pca = weight_pca_analysis.WeightPCAAnalysis(pca_args)

plotting_helpers.figure_width_mm = 180 #final width of the saved pdf
plotting_helpers.figure_fontsize = 5 #pt, all text
weight_pca.plot_pooled_pca_summary_per_pc(image_model='alexnet',style='swap',figsize=(7.1,5.0),pc_label_base=0)
plotting_helpers.figure_width_mm = None
plotting_helpers.figure_fontsize = None
## saves figures/supplemental/pooled_pca_summary_per_pc_swap_alexnet.pdf


## the plotted values (r² of each feature with each of the first 5 pooled PCs, with significance) are saved next to the figure
plotting_helpers.collect_source_data([os.path.join(args_dict['dir'],'figures','supplemental','pooled_pca_summary_per_pc_swap_alexnet.csv')],
                                     os.path.join(args_dict['dir'],'figures','supplemental','sourcedata'),'SupplementaryFigure11')